# AI-Powered Student Performance & Career Analytics

**Project Domain:** Data Analytics with Artificial Intelligence  
**Tools:** Python, Pandas, NumPy, Matplotlib, Scikit-learn

## Project Overview
This project analyzes student academic and technical-skill data using Data Analytics and Machine Learning. It predicts student performance levels, identifies skill gaps, and recommends a suitable career domain based on the available student features.

## 1. Import Libraries and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATA_URL = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition/WA_Fn-UseC_-HR-Employee-Attrition.csv"
# For this student-focused project, use the included local dataset when available.
DATA_PATH = "data/student_performance.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## 2. Dataset Information and Data Quality Check

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include='all').T)
print("Missing values:\n", df.isnull().sum())

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,4))
df['performance_level'].value_counts().plot(kind='bar')
plt.title('Student Performance Distribution')
plt.xlabel('Performance Level')
plt.ylabel('Number of Students')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

skill_cols = ['attendance','assignment_score','internal_marks','python_score','sql_score','ml_score','communication_score']
df[skill_cols].mean().sort_values(ascending=False).plot(kind='bar', figsize=(9,4))
plt.title('Average Student Skill Scores')
plt.ylabel('Average Score')
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 4. Feature Preparation

In [ ]:
features = [
    'attendance','assignment_score','internal_marks','python_score',
    'sql_score','ml_score','communication_score','projects',
    'certifications','internship'
]
X = df[features]
y = df['performance_level']

preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), ['internship'])
], remainder='passthrough')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training records:', len(X_train))
print('Testing records:', len(X_test))

## 5. Train the AI/ML Model

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=250, random_state=42))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
accuracy = accuracy_score(y_test, pred)
print(f'Test Accuracy: {accuracy*100:.2f}%')
print(classification_report(y_test, pred, zero_division=0))

## 6. Performance Prediction for a Student

In [ ]:
sample_student = pd.DataFrame([{
    'attendance': 88,
    'assignment_score': 82,
    'internal_marks': 78,
    'python_score': 90,
    'sql_score': 85,
    'ml_score': 80,
    'communication_score': 70,
    'projects': 3,
    'certifications': 2,
    'internship': 'Yes'
}])
print('Predicted Performance:', model.predict(sample_student)[0])

## 7. AI-Based Career Recommendation

In [ ]:
def career_scores(row):
    return {
        'Data Analyst': 0.40*row['sql_score'] + 0.30*row['python_score'] + 0.20*row['communication_score'] + 0.10*row['assignment_score'],
        'AI/ML Engineer': 0.45*row['ml_score'] + 0.30*row['python_score'] + 0.15*row['internal_marks'] + 0.10*row['projects']*15,
        'Software Developer': 0.40*row['python_score'] + 0.25*row['assignment_score'] + 0.20*row['internal_marks'] + 0.15*row['projects']*15,
        'Business Analyst': 0.35*row['communication_score'] + 0.30*row['sql_score'] + 0.20*row['assignment_score'] + 0.15*row['certifications']*15
    }

scores = career_scores(sample_student.iloc[0])
career_result = pd.DataFrame({'Career': list(scores.keys()), 'Match Score': list(scores.values())}).sort_values('Match Score', ascending=False)
display(career_result)
print('Recommended Career Domain:', career_result.iloc[0]['Career'])

## 8. Skill Gap Analysis

In [ ]:
target = 75
gaps = {}
for col in skill_cols:
    score = float(sample_student.iloc[0][col])
    if score < target:
        gaps[col.replace('_',' ').title()] = target - score

if gaps:
    print('Skills below target:', gaps)
else:
    print('All tracked skills meet the target score.')

## 9. Conclusion
The project demonstrates a complete Data Analytics + AI workflow. Student data is explored and visualized, a Random Forest model predicts performance, and a rule-based weighted matching layer provides career-domain recommendations and skill-gap information.

**Important:** The included dataset is synthetic for demonstration. For institutional use, it should be replaced with properly collected and authorized student data.